In [ ]:
import sys
import json
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import optuna
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ---------- CONFIG ----------
BASE_DIR = Path.cwd().parent.parent
PROCESSED_DIR = BASE_DIR / "bigdata" / "processed"
ARTIFACTS_DIR = BASE_DIR / "artifacts" / "final_models"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

VAL_CUTOFF = "2023-01-01"
TEST_CUTOFF = "2023-06-01"
HORIZON = 20               # дней
N_TRIALS = 50              # для Optuna (можно уменьшить до 15 для скорости)
RANDOM_SEED = 42


d:\Storage\Projects\dpo\dpo\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet(PROCESSED_DIR / "combined_features.parquet")

feature_cols = [line.strip() for line in open(PROCESSED_DIR / "feature_columns.txt") if line.strip()]
for col in tqdm(feature_cols, desc="Clipping outliers"):
    data = df[col].dropna()
    if len(data) == 0:
        continue
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        continue
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    df[col] = df[col].clip(lower, upper)

# Таргеты
target_binary = f"target_binary_{HORIZON}d"
target_return = f"target_return_{HORIZON}d"
assert target_binary in df.columns, f"{target_binary} not found"
assert target_return in df.columns, f"{target_return} not found"

Clipping outliers: 100%|██████████| 109/109 [00:02<00:00, 45.38it/s]


In [ ]:
def temporal_val_split(df, feature_cols, target_col, val_cutoff=VAL_CUTOFF, test_cutoff=TEST_CUTOFF):
    """Корректный временной сплит на train / val / test по дате."""
    train_mask = df["date"] < val_cutoff
    val_mask = (df["date"] >= val_cutoff) & (df["date"] < test_cutoff)
    test_mask = df["date"] >= test_cutoff

    X_train = df.loc[train_mask, feature_cols]
    X_val = df.loc[val_mask, feature_cols]
    X_test = df.loc[test_mask, feature_cols]
    y_train = df.loc[train_mask, target_col]
    y_val = df.loc[val_mask, target_col]
    y_test = df.loc[test_mask, target_col]

    valid_train = y_train.notna()
    valid_val = y_val.notna()
    valid_test = y_test.notna()

    return (X_train[valid_train], X_val[valid_val], X_test[valid_test],
            y_train[valid_train], y_val[valid_val], y_test[valid_test])

: 

In [ ]:
# ---------- БИНАРНАЯ МОДЕЛЬ (HORIZON = 20) ----------
print("\n=== Binary Classification (CatBoost) ===")
X_train_b, X_val_b, X_test_b, y_train_b, y_val_b, y_test_b = temporal_val_split(df, feature_cols, target_binary)
print(f"Train: {X_train_b.shape}, Test: {X_test_b.shape}, Pos rate train: {y_train_b.mean():.3f}")

def objective_binary(trial):
    params = {
        'depth': trial.suggest_int('depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'iterations': trial.suggest_int('iterations', 300, 2500),
        'random_seed': RANDOM_SEED,
        'verbose': False,
        'allow_writing_files': False,
        'task_type': 'GPU'
    }

    model = CatBoostClassifier(**params)
    model.fit(Pool(X_train_b, y_train_b), verbose=False)
    proba = model.predict_proba(X_val_b)[:, 1]
    return roc_auc_score(y_val_b, proba)

study_b = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_b.optimize(objective_binary, n_trials=N_TRIALS, show_progress_bar=True)
print("Best params:", study_b.best_params)

# Финальное обучение на всём train
final_model_b = CatBoostClassifier(**study_b.best_params, random_seed=RANDOM_SEED, verbose=False, task_type='GPU')
X_tr_b = pd.concat([X_train_b, X_val_b], axis=0)
y_tr_b = pd.concat([y_train_b, y_val_b], axis=0)
final_model_b.fit(Pool(X_tr_b, y_tr_b), verbose=False)
y_pred_proba = final_model_b.predict_proba(X_test_b)[:, 1]
y_pred_bin = (y_pred_proba >= 0.5).astype(int)
test_auc = roc_auc_score(y_test_b, y_pred_proba)
print(f"Test AUC: {test_auc:.4f}")

# Сохранение
model_path_b = ARTIFACTS_DIR / f"catboost_binary_{HORIZON}d.cbm"
final_model_b.save_model(str(model_path_b))
print(f"Binary model saved to {model_path_b}")


=== Binary Classification (CatBoost) ===


[I 2026-05-21 14:40:20,946] A new study created in memory with name: no-name-6842de8d-89ab-488e-8bac-c517e4151a23


Train: (666251, 109), Test: (76054, 109), Pos rate train: 0.490


Best trial: 0. Best value: 0.508498:   2%|▏         | 1/50 [00:33<27:27, 33.62s/it]

[I 2026-05-21 14:40:54,565] Trial 0 finished with value: 0.5084978984340061 and parameters: {'depth': 7, 'learning_rate': 0.07969454818643935, 'l2_leaf_reg': 0.8471801418819978, 'iterations': 1617}. Best is trial 0 with value: 0.5084978984340061.


...

[I 2026-05-21 14:48:39,030] Trial 8 finished with value: 0.5845597619194788 and parameters: {'depth': 3, 'learning_rate': 0.07902619549708234, 'l2_leaf_reg': 7.2866537374910445, 'iterations': 2079}. Best is trial 8 with value: 0.5845597619194788.


Best trial: 8. Best value: 0.58456:  20%|██        | 10/50 [08:41<23:51, 35.80s/it]

[I 2026-05-21 14:49:02,014] Trial 9 finished with value: 0.4556217208480437 and parameters: {'depth': 6, 'learning_rate': 0.0015679933916723015, 'l2_leaf_reg': 0.5456725485601477, 'iterations': 1268}. Best is trial 8 with value: 0.5845597619194788.


Best trial: 8. Best value: 0.58456:  22%|██▏       | 11/50 [09:03<20:40, 31.81s/it]

[I 2026-05-21 14:49:24,771] Trial 10 finished with value: 0.5029573769329498 and parameters: {'depth': 3, 'learning_rate': 0.00884412033211848, 'l2_leaf_reg': 7.553503645583182, 'iterations': 1862}. Best is trial 8 with value: 0.5845597619194788.


In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train_b_imp = pd.DataFrame(imputer.fit_transform(X_train_b), columns=X_train_b.columns)
X_val_b_imp = pd.DataFrame(imputer.transform(X_val_b), columns=X_val_b.columns)
X_test_b_imp = pd.DataFrame(imputer.transform(X_test_b), columns=X_test_b.columns)

# Pipeline: масштабирование + логистическая регрессия
logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_SEED,
        class_weight='balanced',  # Балансировка классов
        solver='lbfgs'
    ))
])

logreg.fit(X_train_b_imp, y_train_b)

# Оценка
y_pred_proba_lr = logreg.predict_proba(X_test_b_imp)[:, 1]
y_pred_lr = logreg.predict(X_test_b_imp)
test_auc_lr = roc_auc_score(y_test_b, y_pred_proba_lr)
test_acc_lr = accuracy_score(y_test_b, y_pred_lr)

print(f"Test AUC: {test_auc_lr:.4f}")
print(f"Test Accuracy: {test_acc_lr:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_b, y_pred_lr))

Test AUC: 0.6187
Test Accuracy: 0.5905

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.79      0.69     43492
           1       0.54      0.33      0.41     32562

    accuracy                           0.59     76054
   macro avg       0.57      0.56      0.55     76054
weighted avg       0.58      0.59      0.57     76054



In [ ]:
# ---------- РЕГРЕССИОННАЯ МОДЕЛЬ (HORIZON = 20) – опционально ----------
print("\n=== Regression (CatBoost) ===")
X_train_r, X_val_r, X_test_r, y_train_r, y_val_r, y_test_r = temporal_val_split(df, feature_cols, target_return)
print(f"Train: {X_train_r.shape}, Test: {X_test_r.shape}, Mean target: {y_train_r.mean():.4f}")

def objective_return(trial):
    params = {
        'depth': trial.suggest_int('depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'iterations': trial.suggest_int('iterations', 300, 1500),
        'random_seed': RANDOM_SEED,
        'verbose': False,
        'allow_writing_files': False,
        'task_type': 'GPU',
        'eval_metric': 'RMSE',
        'loss_function': 'RMSE'
    }

    model = CatBoostRegressor(**params)
    model.fit(Pool(X_train_r, y_train_r), verbose=False, early_stopping_rounds=50)
    pred = model.predict(X_val_r)
    return -np.sqrt(mean_squared_error(y_val_r, pred))

study_r = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_r.optimize(objective_return, n_trials=N_TRIALS, show_progress_bar=True)
print("Best params:", study_r.best_params)

final_model_r = CatBoostRegressor(**study_r.best_params, random_seed=RANDOM_SEED, verbose=False, task_type='GPU')
X_tr_r = pd.concat([X_train_r, X_val_r], axis=0)
y_tr_r = pd.concat([y_train_r, y_val_r], axis=0)
final_model_r.fit(Pool(X_tr_r, y_tr_r), verbose=False)
y_pred_r = final_model_r.predict(X_test_r)
test_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
print(f"Test RMSE: {test_rmse:.4f}, R2: {final_model_r.score(X_test_r, y_test_r):.4f}")

model_path_r = ARTIFACTS_DIR / f"catboost_return_{HORIZON}d.cbm"
final_model_r.save_model(str(model_path_r))
print(f"Regression model saved to {model_path_r}")


=== Regression (CatBoost) ===


[I 2026-05-21 13:36:55,069] A new study created in memory with name: no-name-4978635a-23e2-4896-ac4a-09c531f31953


Train: (666171, 109), Test: (71171, 109), Mean target: 0.0033


Best trial: 0. Best value: -0.183818:   2%|▏         | 1/50 [00:13<11:01, 13.49s/it]

[I 2026-05-21 13:37:08,562] Trial 0 finished with value: -0.1838175525752691 and parameters: {'depth': 6, 'learning_rate': 0.07969454818643935, 'l2_leaf_reg': 0.8471801418819978, 'iterations': 1018}. Best is trial 0 with value: -0.1838175525752691.


...

[I 2026-05-21 13:53:07,311] Trial 47 finished with value: -0.16898965344437408 and parameters: {'depth': 12, 'learning_rate': 0.0018774397260879628, 'l2_leaf_reg': 5.947978683243684, 'iterations': 1342}. Best is trial 26 with value: -0.16082854341835734.


Best trial: 26. Best value: -0.160829:  98%|█████████▊| 49/50 [16:26<00:30, 30.94s/it]

[I 2026-05-21 13:53:21,562] Trial 48 finished with value: -0.16447506883771193 and parameters: {'depth': 9, 'learning_rate': 0.0012120211229306012, 'l2_leaf_reg': 0.23127263042876703, 'iterations': 565}. Best is trial 26 with value: -0.16082854341835734.


Best trial: 26. Best value: -0.160829: 100%|██████████| 50/50 [17:08<00:00, 20.58s/it]


[I 2026-05-21 13:54:03,915] Trial 49 finished with value: -0.17183735225895813 and parameters: {'depth': 11, 'learning_rate': 0.0035241865680747417, 'l2_leaf_reg': 0.42280743044935626, 'iterations': 902}. Best is trial 26 with value: -0.16082854341835734.
Best params: {'depth': 12, 'learning_rate': 0.001563838044384607, 'l2_leaf_reg': 0.10153533180366722, 'iterations': 305}
Test RMSE: 0.1742, R2: -0.0020
Regression model saved to d:\Storage\Projects\dpo\dpo\project\artifacts\final_models\catboost_return_20d.cbm


In [ ]:
X_train_r_filled = X_train_r.fillna(X_train_r.median())
X_val_r_filled = X_val_r.fillna(X_train_r.median())
X_test_r_filled = X_test_r.fillna(X_train_r.median())

# Pipeline: масштабирование + Ridge регрессия
ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(
        alpha=1.0,
        random_state=RANDOM_SEED
    ))
])

ridge.fit(X_train_r_filled, y_train_r)

# Оценка
y_pred_ridge = ridge.predict(X_test_r_filled)
test_rmse_ridge = np.sqrt(mean_squared_error(y_test_r, y_pred_ridge))
test_r2_ridge = r2_score(y_test_r, y_pred_ridge)

print(f"Test RMSE: {test_rmse_ridge:.4f}")
print(f"Test R2: {test_r2_ridge:.4f}")

Test RMSE: 0.1724
Test R2: 0.0191
